# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM (Large Language Model) este un tip de inteligență artificială antrenată pe cantități masive de text, permițându-i să înțeleagă, să genereze și să traducă limbajul uman cu o fluiditate remarcabilă. Această capacitate îi permite să efectueze o varietate largă de sarcini, de la răspunsuri la întrebări și rezumate, la scriere creativă și generare de cod.
Un LLM (Large Language Model) este un tip de model lingvistic bazat pe rețele neuronale profunde, antrenat pe cantități masive de date textuale pentru a înțelege, genera și manipula limbajul uman. Aceste modele pot realiza o varietate de sarcini, de la răspunsuri la întrebări și rezumare de text, până la traducere și creare de conținut creativ.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [6]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 5 ani.
Maximum 80 de cuvinte.
Răspunde anti-suveranist, defensiv.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
În ultimii cinci ani, politica românească a fost marcată de o instabilitate guvernamentală frecventă și de o polarizare crescândă, alimentată de dezbateri interne și externe. Aceste dinamici au necesitat o consolidare a instituțiilor democratice și o cooperare internațională sporită pentru a asigura stabilitatea și progresul țării.

--- Gemini 2.5 Flash ---
Pentru a asigura stabilitatea pro-europeană și a contracara ascensiunea populismului, scena politică românească a fost marcată de formarea unei coaliții guvernamentale largi, fără precedent, între partide tradițional rivale. Această alianță a vizat consolidarea parcursului european al țării și implementarea reformelor necesare, în ciuda provocărilor interne și a presiunilor externe, menținând angajamentul ferm față de valorile democratice și statul de drept.

--- OpenRouter Free ---
Principala schimbare din politica românească a ultimilor 5 ani a fost ascensiunea curentelor suveraniste, care au contest

## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [7]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Adevărata putere modernă nu stă în izolarea granițelor, ci în greutatea deciziilor luate la mesele unde se împarte viitorul lumii."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Diplomatic, analitic.
Emoție dominantă: Încredere, convingere.
Țintă principală: Liderii politici și decidenții globali.
Populism: nu

--- Gemini 2.5 Flash ---
Ton: Analitic-strategie
Emoție dominantă: Convingere
Țintă principală: Mentalitatea izolaționistă
Populism: nu

--- OpenRouter Free ---
Comentariul spune că politica modernă este în interacție cu viitorul.  
Acesta evaluează o tendere de populism.  
Ne poate fi un sentiment negativ.  
Suntem în greutatea deciziilor de la un moment.


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [8]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [9]:
COMENTARIU = "Adevărata putere modernă nu stă în izolarea granițelor, ci în greutatea deciziilor luate la mesele unde se împarte viitorul lumii."
SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'neutru', 'emotie_dominanta': 'neutru', 'tinta_principala': 'puterea mondiala', 'populism': False, 'explicatie_scurta': 'Comentariul sugerează că puterea reală în lumea modernă nu provine din controlul granițelor, ci din participarea la deciziile globale care modelează viitorul.'}

--- Gemini 2.5 Flash ---
{'ton': 'pozitiv', 'emotie_dominanta': 'speranta', 'tinta_principala': 'politicile izolaționiste', 'populism': False, 'explicatie_scurta': 'Comentariul definește puterea modernă prin influența globală și deciziile colective, criticând implicit izolarea și promovând o viziune de colaborare.'}

--- OpenRouter Free ---
{'emotie_dominanta': 'furie', 'explicatie_scurta': 'Comentariul exprimă o critică subtilă la adresa politicilor de închidere a granițelor (izolare), sugerând că acestea sunt ineficiente sau periferice față de adevăratul mecanism de exercitare a puterii globale. Tonul este polemic, dar rațional, concentrat pe mecanismele decizionale e

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [10]:
PROMPT_STAB = """
Adevărata putere modernă nu stă în izolarea granițelor, ci în greutatea deciziilor luate la mesele unde se împarte viitorul lumii..
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Această afirmație sugerează că influența politică reală se manifestă prin participarea la forumuri internaționale unde se stabilesc politici globale, mai degrabă decât prin politici naționale izolate. Astfel, deciziile luate în aceste cercuri influențează direct cursul evenimentelor la nivel mondial, având un impact semnificativ asupra vieții politice interne a statelor.

temperature=0.7:
Această afirmație sugerează că influența politică actuală este determinată mai degrabă de participarea la procesele decizionale globale, decât de controlul strict al granițelor naționale. Astfel, pentru viața politică, aceasta implică o concentrare pe diplomație, colaborare internațională și participarea activă la forumuri multilaterale pentru a modela direcția globală.

temperature=1.2:
Această afirmație sugerează că deciziile politice cruciale pentru viitorul global nu mai sunt luate doar la n

## 7. Alegerea modelului pentru proiect
### Decizie
**Model principal ales:** Gemini 2.5 Flash 
**Model de rezervă:**  Gemini 2.5 Flash Lite
**Temperature recomandată:**  0.7
**De ce am ales acest model?**  
Mult mai coerent și accurate pe informații, analizeaza mai concis emoția și tonul. Este mult mai ok pentru exprimarea în limba română.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [11]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.7

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales